# Example workflow for SOM integration into StatMagic HMI


### Different txt and dictionary files

- bmu_ids.txt
- cluster_hit_count.txt
- result_som.txt
- RunStats.txt
- cluster_label_counts.txt
- geo_labeled_bmu.txt
- labels_flat.txt
- labels_grouped.txt
- **result_geo.txt** 🚨: these are important for plotting, but can become super-large.
- som.dictionary: looks like a binary file
- cluster.dictionary: looks like a binary file

### Plots

- db_score: Davies-Bouldin Index (related to kMeans clustering)
- bmu_id_mesh: SOM grid
- cluster_hit_count: number of pixels per cluster
- cluster_label_count: number of labels per cluster
- geoplots: depending on number of input layers
- boxplots: depending on number of input layers
- somplots: depending on number of input layers

### Folders
- GeoTIFF: **contains all tif outputs, including**
    - Discretized input layers (I think these are some kind of codebook vector maps), files starting with **“b_”**
    - **q_error**
    - **bmu_id**
    - **cluster**
    - **bmu_bmu_label_count**
    - **bmu_cluster_label_count**

### Reminder
Some of these outputs are only relevant if
- kmeans clustering was performed
- labels were provided
- plotting was activated

## Imports

In [24]:
import os
import pandas as pd

from beak.integration.statmagic.call_som import run_som
from beak.integration.statmagic.utils import (
    _get_data_folder,
    _extract_rows,
    _download_cdr_files,
    _filter_files, 
)


## Block 1: Get data folder and JSON file

In [32]:
# 1. Define main data folder: we use the beak/data folder provided by the MTRI package template
data_folder = _get_data_folder()

# 2. Select JSON and read data: use a collective folder for all JSON files
file_path = data_folder / "cdr" / "json" / "dummy_model_run_som.json"
json_data = pd.read_json(file_path)
payload = json_data.loc["payload"]

# 3. Get CMA info etc.: the model_run_id is used as identifier for the subfolder to download files and save model outputs
model_run_id = payload["model_run_id"]
cma_id = payload["cma_id"]

# 4. Extract evidence layers and labels
evidence_layers = payload["event"]["evidence_layers"]
evidence_layers = pd.json_normalize(evidence_layers)
evidence_layers = evidence_layers[~evidence_layers["label_raster"]]

labels = payload["event"]["evidence_layers"]
labels = pd.json_normalize(labels)
labels = labels[labels["label_raster"]]

# 5. Extract download url's from evidence layers: get the list of all urls in the payload
download_urls_layers = _extract_rows(data=evidence_layers, attribute="download_url")
download_urls_labels = _extract_rows(data=labels, attribute="download_url")

# 6. Filter tif files: only keep .tif/.tiff files left for downloading
filtered_urls_layers = _filter_files(file_list=download_urls_layers, file_suffix=(".tif", ".tiff"), file_prefix=None)
filtered_urls_labels = _filter_files(file_list=download_urls_labels, file_suffix=(".tif", ".tiff"), file_prefix=None)

# 7. Download files and create file list to be used as input file list for the SOM call
identifier = cma_id
base_folder = data_folder / "cdr" / "test" / identifier

download_folder_layers = base_folder / "inputs"
download_folder_labels = base_folder / "labels"
output_folder = base_folder / "outputs" / "som"

labels_name = "EPSG_102008_RES_2240_us_LAWLEY22_HEX.0.1.tif"

print(f"Download folder: {download_folder_layers}")
print(f"Output folder: {output_folder}")

Download folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\cdr\test\dummy_model_run\inputs
Output folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\cdr\test\dummy_model_run\outputs\som


In [29]:
use_local_files = True

if use_local_files is True:
    input_file_list = [os.path.join(download_folder_layers, file) for file in os.listdir(download_folder_layers)]
    input_labels = os.path.join(download_folder_labels, labels_name)
else:
    input_file_list = _download_cdr_files(
        download_urls=filtered_urls_layers,
        download_folder=download_folder_layers,
        verify_ssl=False,
    )

    input_labels = _download_cdr_files(
        download_urls=filtered_urls_labels,
        download_folder=download_folder_layers,
        verify_ssl=False,
    )


## Block 2: Run the SOM algorithm

In [ ]:
# 1. Run SOM algorithm
prospectivity_output_layers = run_som(
    input_layers=input_file_list,
    input_labels=input_labels,
    config_file=file_path,
    output_folder=output_folder
)
